# 천안시 m단위 격자별 데이터 결합 파이프라인

격자 크기(`GRID_M`)만 바꿔서 500m, 750m, 1000m 등으로 다시 만들 수 있게 정리한 버전

1. 이미 위경도가 있는 상권/상가, 병원, 약국 데이터를 격자에 매칭
2. 학교, 복지시설, 행정기관, 착한가격업소 등 주소 기반 시설은 지오코딩 결과의 경도/위도를 이용해 격자에 매칭


## 1. 설정

- `GRID_M`만 바꾸면 다른 격자 크기로 재생성
- `GIMI9_XLSX`는 새 지오코딩 결과 엑셀을 다시 반영할 때만 지정
- 이미 `cheonan_address_geocoding_cache.csv`가 있으면 `GIMI9_XLSX = None`으로 둬도 됨


In [ ]:
from pathlib import Path

GRID_M = 750  # 500, 750, 1000 등 원하는 m 단위로 변경

WORKSPACE = Path("/content/drive/MyDrive/Cheonan")
OUTPUT_DIR = WORKSPACE
BOUNDARY_ZIP = WORKSPACE / "cheonan_adm_dong_boundary_20250630.zip"

HYUKJIN_DIR = Path(r"C:\\Users\\jin57\\Documents\\GitHub\\Cheonan-Data-Analysis\\Hyukjin")
HYEONSEOK_DIR = Path(r"C:\\Users\\jin57\\Documents\\GitHub\\Cheonan-Data-Analysis\\Hyeonseok")

ADDRESS_FACILITY_ROWS = OUTPUT_DIR / "cheonan_address_facility_rows.csv"
ADDRESS_GEOCODE_CACHE = OUTPUT_DIR / "cheonan_address_geocoding_cache.csv"

# 새 GIMI9 결과 엑셀을 바로 반영하려면 아래 경로를 넣으세요. 이미 cache가 있으면 None.
GIMI9_XLSX = None
# GIMI9_XLSX = Path(r"C:\\Users\\jin57\\Downloads\\GIMI9_GEOCODE_20260703_135519_cheonan_address_geocoding_tasks.xlsx")


## 2. 함수 정의

아래 셀은 실제 처리 함수
경계 ZIP에서 격자를 만들고, 좌표 데이터와 지오코딩 결과를 `grid_id` 기준으로 집계


In [ ]:
# -*- coding: utf-8 -*-
"""Reusable Cheonan grid-feature pipeline.

Inputs assumed:
  - Cheonan administrative boundary ZIP:
      outputs/cheonan_adm_dong_boundary_20250630.zip
  - Team data folders with coordinate CSVs:
      .../Cheonan-Data-Analysis/Hyeonseok
      .../Cheonan-Data-Analysis/Hyukjin
  - Optional address-geocoding result:
      outputs/cheonan_address_facility_rows.csv
      outputs/cheonan_address_geocoding_cache.csv
    or a GIMI9 geocoding result XLSX passed by --gimi9-xlsx

Example:
  python work/cheonan_grid_pipeline.py --grid-m 750
"""

from __future__ import annotations

import argparse
import json
import math
import re
import struct
import sys
import unicodedata
import zipfile
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd


WORKSPACE = Path(r"C:\Users\jin57\Documents\Codex\2026-06-23\m")
OUTPUT_DIR = WORKSPACE / "outputs"
BOUNDARY_ZIP = OUTPUT_DIR / "cheonan_adm_dong_boundary_20250630.zip"
HYUKJIN_DIR = Path(r"C:\Users\jin57\Documents\GitHub\Cheonan-Data-Analysis\Hyukjin")
HYEONSEOK_DIR = Path(r"C:\Users\jin57\Documents\GitHub\Cheonan-Data-Analysis\Hyeonseok")

ADDRESS_FACILITY_ROWS = OUTPUT_DIR / "cheonan_address_facility_rows.csv"
ADDRESS_GEOCODE_CACHE = OUTPUT_DIR / "cheonan_address_geocoding_cache.csv"


def safe_name(value: object, max_len: int = 48) -> str:
    text = unicodedata.normalize("NFC", str(value).strip())
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"[^\w]+", "_", text, flags=re.UNICODE)
    text = re.sub(r"_+", "_", text).strip("_")
    return (text or "unknown")[:max_len]


def read_csv_any(path: Path, nrows: int | None = None) -> tuple[pd.DataFrame, str]:
    last_error: Exception | None = None
    for enc in ("utf-8-sig", "utf-8", "cp949", "euc-kr"):
        try:
            return pd.read_csv(path, encoding=enc, nrows=nrows), enc
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Could not read CSV {path}: {last_error}")


def dbf_records(dbf_bytes: bytes, encoding: str = "cp949") -> list[dict[str, str]]:
    nrec = struct.unpack("<I", dbf_bytes[4:8])[0]
    header_len = struct.unpack("<H", dbf_bytes[8:10])[0]
    rec_len = struct.unpack("<H", dbf_bytes[10:12])[0]
    fields: list[tuple[str, int]] = []
    off = 32
    while dbf_bytes[off] != 0x0D:
        desc = dbf_bytes[off : off + 32]
        name = desc[:11].split(b"\x00", 1)[0].decode("ascii", "ignore")
        size = desc[16]
        fields.append((name, size))
        off += 32
    rows = []
    for i in range(nrec):
        row = dbf_bytes[header_len + i * rec_len : header_len + (i + 1) * rec_len]
        pos = 1
        values = {}
        for name, size in fields:
            raw = row[pos : pos + size]
            pos += size
            values[name] = raw.decode(encoding, "ignore").strip()
        rows.append(values)
    return rows


def read_boundary_shapes(boundary_zip: Path = BOUNDARY_ZIP) -> list[dict[str, object]]:
    with zipfile.ZipFile(boundary_zip) as z:
        shp_name = next(n for n in z.namelist() if n.endswith(".shp"))
        dbf_name = next(n for n in z.namelist() if n.endswith(".dbf"))
        shp_bytes = z.read(shp_name)
        attrs = dbf_records(z.read(dbf_name), "cp949")

    shapes: list[dict[str, object]] = []
    off = 100
    while off + 8 <= len(shp_bytes):
        rec_no, content_words = struct.unpack(">2i", shp_bytes[off : off + 8])
        off += 8
        content = memoryview(shp_bytes)[off : off + content_words * 2]
        off += content_words * 2
        if len(content) < 44 or struct.unpack("<i", content[0:4])[0] != 5:
            continue
        bbox = struct.unpack("<4d", content[4:36])
        num_parts, num_points = struct.unpack("<2i", content[36:44])
        part_start = 44
        parts = list(struct.unpack("<" + "i" * num_parts, content[part_start : part_start + 4 * num_parts]))
        point_start = part_start + 4 * num_parts
        points = [
            struct.unpack("<2d", content[point_start + 16 * i : point_start + 16 * (i + 1)])
            for i in range(num_points)
        ]
        rings = []
        for i, start in enumerate(parts):
            end = parts[i + 1] if i + 1 < len(parts) else num_points
            ring = points[start:end]
            if len(ring) < 3:
                continue
            xs = [p[0] for p in ring]
            ys = [p[1] for p in ring]
            rings.append({"points": ring, "bbox": (min(xs), min(ys), max(xs), max(ys))})
        shapes.append({"attrs": attrs[rec_no - 1], "bbox": bbox, "rings": rings})
    return shapes


def point_in_ring(x: float, y: float, ring: list[tuple[float, float]]) -> bool:
    inside = False
    j = len(ring) - 1
    for i, (xi, yi) in enumerate(ring):
        xj, yj = ring[j]
        if (yi > y) != (yj > y):
            x_inter = (xj - xi) * (y - yi) / ((yj - yi) or 1e-30) + xi
            if x < x_inter:
                inside = not inside
        j = i
    return inside


def point_in_shape(x: float, y: float, shape: dict[str, object]) -> bool:
    minx, miny, maxx, maxy = shape["bbox"]  # type: ignore[misc]
    if x < minx or x > maxx or y < miny or y > maxy:
        return False
    inside = False
    for ring in shape["rings"]:  # type: ignore[index]
        rx0, ry0, rx1, ry1 = ring["bbox"]
        if x < rx0 or x > rx1 or y < ry0 or y > ry1:
            continue
        if point_in_ring(x, y, ring["points"]):
            inside = not inside
    return inside


def find_admin_for_point(x: float, y: float, shapes: list[dict[str, object]]) -> dict[str, str] | None:
    for shape in shapes:
        if point_in_shape(x, y, shape):
            return shape["attrs"]  # type: ignore[return-value]
    return None


def create_grid(grid_m: int, shapes: list[dict[str, object]]) -> pd.DataFrame:
    all_x = []
    all_y = []
    for shape in shapes:
        for ring in shape["rings"]:  # type: ignore[index]
            for x, y in ring["points"]:
                all_x.append(x)
                all_y.append(y)

    minx, maxx = min(all_x), max(all_x)
    miny, maxy = min(all_y), max(all_y)
    x0 = math.floor(minx / grid_m) * grid_m
    x1 = math.ceil(maxx / grid_m) * grid_m
    y0 = math.floor(miny / grid_m) * grid_m
    y1 = math.ceil(maxy / grid_m) * grid_m

    rows = []
    y = y0
    while y < y1:
        x = x0
        while x < x1:
            cx = x + grid_m / 2
            cy = y + grid_m / 2
            admin = find_admin_for_point(cx, cy, shapes)
            if admin is not None:
                rows.append(
                    {
                        "grid_id": f"C{len(rows) + 1:05d}",
                        "x_min": float(x),
                        "y_min": float(y),
                        "x_max": float(x + grid_m),
                        "y_max": float(y + grid_m),
                        "center_x": float(cx),
                        "center_y": float(cy),
                        "grid_m": grid_m,
                        "selection": "center_in_cheonan",
                        "center_adm_cd": admin.get("ADM_CD", ""),
                        "center_adm_nm": admin.get("ADM_NM", ""),
                    }
                )
            x += grid_m
        y += grid_m
    return pd.DataFrame(rows)


def lonlat_to_kgd2002_central(lon: np.ndarray, lat: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    a = 6378137.0
    inv_f = 298.257222101
    f = 1.0 / inv_f
    e2 = f * (2.0 - f)
    ep2 = e2 / (1.0 - e2)
    lon0 = math.radians(127.0)
    lat0 = math.radians(38.0)
    fe = 200000.0
    fn = 600000.0
    k0 = 1.0

    phi = np.radians(lat.astype(float))
    lam = np.radians(lon.astype(float))

    def meridian_arc(phi_value: np.ndarray | float) -> np.ndarray | float:
        e4 = e2 * e2
        e6 = e4 * e2
        return a * (
            (1 - e2 / 4 - 3 * e4 / 64 - 5 * e6 / 256) * phi_value
            - (3 * e2 / 8 + 3 * e4 / 32 + 45 * e6 / 1024) * np.sin(2 * phi_value)
            + (15 * e4 / 256 + 45 * e6 / 1024) * np.sin(4 * phi_value)
            - (35 * e6 / 3072) * np.sin(6 * phi_value)
        )

    sin_phi = np.sin(phi)
    cos_phi = np.cos(phi)
    tan_phi = np.tan(phi)
    n = a / np.sqrt(1 - e2 * sin_phi * sin_phi)
    t = tan_phi * tan_phi
    c = ep2 * cos_phi * cos_phi
    aa = (lam - lon0) * cos_phi
    m = meridian_arc(phi)
    m0 = meridian_arc(lat0)
    x = fe + k0 * n * (
        aa
        + (1 - t + c) * aa**3 / 6
        + (5 - 18 * t + t**2 + 72 * c - 58 * ep2) * aa**5 / 120
    )
    y = fn + k0 * (
        m
        - m0
        + n
        * tan_phi
        * (
            aa**2 / 2
            + (5 - t + 9 * c + 4 * c**2) * aa**4 / 24
            + (61 - 58 * t + t**2 + 600 * c - 330 * ep2) * aa**6 / 720
        )
    )
    return x, y


def grid_lookup(grid: pd.DataFrame) -> dict[tuple[int, int], str]:
    return {
        (int(round(row.x_min)), int(round(row.y_min))): row.grid_id
        for row in grid[["grid_id", "x_min", "y_min"]].itertuples(index=False)
    }


def assign_lonlat_to_grid(
    df: pd.DataFrame,
    lon_col: str,
    lat_col: str,
    grid_m: int,
    lookup: dict[tuple[int, int], str],
) -> pd.DataFrame:
    lon = pd.to_numeric(df[lon_col], errors="coerce")
    lat = pd.to_numeric(df[lat_col], errors="coerce")
    valid = lon.between(120, 135) & lat.between(30, 40)
    out = df.loc[valid].copy()
    if out.empty:
        return pd.DataFrame()
    x, y = lonlat_to_kgd2002_central(lon.loc[valid].to_numpy(), lat.loc[valid].to_numpy())
    x_min = np.floor(x / grid_m) * grid_m
    y_min = np.floor(y / grid_m) * grid_m
    out["_lon"] = lon.loc[valid].to_numpy()
    out["_lat"] = lat.loc[valid].to_numpy()
    out["_x"] = x
    out["_y"] = y
    out["_x_min"] = x_min
    out["_y_min"] = y_min
    out["grid_id"] = [lookup.get((int(round(xm)), int(round(ym))), "") for xm, ym in zip(x_min, y_min)]
    return out


def aggregate_coordinate_sources(grid: pd.DataFrame, grid_m: int) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    lookup = grid_lookup(grid)
    feature_tables = []
    assignments = []
    summary_rows = []

    specs = [
        {
            "source_id": "commerce",
            "path": HYEONSEOK_DIR / "commerce_cheonan_202603.csv",
            "lon_col": "경도",
            "lat_col": "위도",
            "name_col": "상호명",
        },
        {
            "source_id": "hospital",
            "path": HYEONSEOK_DIR / "hospital_cheonan_202603.csv",
            "lon_col": "좌표(X)",
            "lat_col": "좌표(Y)",
            "name_col": "요양기관명",
        },
        {
            "source_id": "pharmacy",
            "path": HYEONSEOK_DIR / "pharmacy_cheonan_202603.csv",
            "lon_col": "좌표(X)",
            "lat_col": "좌표(Y)",
            "name_col": "약국명",
        },
    ]

    for spec in specs:
        path = spec["path"]
        if not path.exists():
            summary_rows.append({"source_id": spec["source_id"], "file_name": path.name, "status": "missing"})
            continue
        df, enc = read_csv_any(path)
        matched = assign_lonlat_to_grid(df, spec["lon_col"], spec["lat_col"], grid_m, lookup)
        if matched.empty:
            summary_rows.append({"source_id": spec["source_id"], "file_name": path.name, "status": "no_valid_coordinates"})
            continue
        matched["source_id"] = spec["source_id"]
        matched["source_file"] = path.name
        matched["source_row"] = matched.index + 2
        matched["name"] = matched[spec["name_col"]] if spec["name_col"] in matched.columns else ""
        assignments.append(
            matched[
                [
                    "source_id",
                    "source_file",
                    "source_row",
                    "grid_id",
                    "name",
                    "_lon",
                    "_lat",
                    "_x",
                    "_y",
                ]
            ].rename(columns={"_lon": "lon", "_lat": "lat", "_x": "x", "_y": "y"})
        )

        ok = matched[matched["grid_id"].astype(str).ne("")]
        features = ok.groupby("grid_id").size().rename(f"{spec['source_id']}_total").reset_index()

        if spec["source_id"] == "commerce":
            for col, prefix in [("상권업종대분류명", "commerce_major"), ("상권업종중분류명", "commerce_middle")]:
                if col in ok.columns:
                    pivot = (
                        ok.pivot_table(index="grid_id", columns=col, values="source_row", aggfunc="count", fill_value=0)
                        .rename(columns=lambda v: f"{prefix}_{safe_name(v)}")
                        .reset_index()
                    )
                    features = features.merge(pivot, on="grid_id", how="left")
        elif spec["source_id"] == "hospital":
            if "종별코드명" in ok.columns:
                pivot = (
                    ok.pivot_table(index="grid_id", columns="종별코드명", values="source_row", aggfunc="count", fill_value=0)
                    .rename(columns=lambda v: f"hospital_type_{safe_name(v)}")
                    .reset_index()
                )
                features = features.merge(pivot, on="grid_id", how="left")
            if "총의사수" in ok.columns:
                s = pd.to_numeric(ok["총의사수"], errors="coerce").fillna(0)
                doctor = ok.assign(_num=s).groupby("grid_id")["_num"].sum().rename("hospital_doctor_sum").reset_index()
                features = features.merge(doctor, on="grid_id", how="left")

        feature_tables.append(features)
        summary_rows.append(
            {
                "source_id": spec["source_id"],
                "file_name": path.name,
                "encoding": enc,
                "status": "processed",
                "rows": int(len(df)),
                "valid_coordinate_rows": int(len(matched)),
                "matched_grid_rows": int(ok.shape[0]),
                "unmatched_grid_rows": int(len(matched) - ok.shape[0]),
            }
        )

    features_all = grid[["grid_id"]].copy()
    for feature in feature_tables:
        features_all = features_all.merge(feature, on="grid_id", how="left")
    for col in features_all.columns:
        if col != "grid_id":
            features_all[col] = pd.to_numeric(features_all[col], errors="coerce").fillna(0)
    assignments_all = pd.concat(assignments, ignore_index=True, sort=False) if assignments else pd.DataFrame()
    return features_all, assignments_all, pd.DataFrame(summary_rows)


def import_gimi9_to_cache(gimi9_xlsx: Path, cache_path: Path = ADDRESS_GEOCODE_CACHE) -> pd.DataFrame:
    result = pd.read_excel(gimi9_xlsx)
    required = {"address_key", "성공여부", "X좌표", "Y좌표"}
    missing = sorted(required - set(result.columns))
    if missing:
        raise ValueError(f"GIMI9 result is missing required columns: {missing}")
    rows = []
    for _, item in result.iterrows():
        ok = str(item.get("성공여부", "")).strip() == "성공"
        lon = pd.to_numeric(pd.Series([item.get("X좌표")]), errors="coerce").iloc[0]
        lat = pd.to_numeric(pd.Series([item.get("Y좌표")]), errors="coerce").iloc[0]
        rows.append(
            {
                "address_key": str(item.get("address_key", "")).strip(),
                "provider": "gimi9",
                "status": "ok" if ok and pd.notna(lon) and pd.notna(lat) else "not_found",
                "matched_query": str(item.get("queries", "")).split(" || ")[0],
                "lon": lon,
                "lat": lat,
                "score_or_type": str(item.get("pos_cd", "")),
                "raw_name": str(item.get("행정표준 행정동명", "")),
                "message": "" if ok else str(item.get("오류메시지", "")),
            }
        )
    imported = pd.DataFrame(rows)
    if cache_path.exists():
        cache = pd.read_csv(cache_path, encoding="utf-8-sig")
        cache = cache[~((cache["provider"].eq("gimi9")) & cache["address_key"].isin(imported["address_key"]))]
        cache = pd.concat([cache, imported], ignore_index=True)
    else:
        cache = imported
    cache.to_csv(cache_path, index=False, encoding="utf-8-sig")
    return cache


def aggregate_address_sources(
    grid: pd.DataFrame,
    grid_m: int,
    gimi9_xlsx: Path | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if gimi9_xlsx is not None:
        cache = import_gimi9_to_cache(gimi9_xlsx)
    elif ADDRESS_GEOCODE_CACHE.exists():
        cache = pd.read_csv(ADDRESS_GEOCODE_CACHE, encoding="utf-8-sig")
    else:
        return grid[["grid_id"]].copy(), pd.DataFrame(), pd.DataFrame([{"status": "missing_geocode_cache"}])

    if not ADDRESS_FACILITY_ROWS.exists():
        return grid[["grid_id"]].copy(), pd.DataFrame(), pd.DataFrame([{"status": "missing_address_facility_rows"}])

    facility = pd.read_csv(ADDRESS_FACILITY_ROWS, encoding="utf-8-sig")
    lookup = grid_lookup(grid)
    ok_cache = cache[cache["status"].eq("ok")].sort_values("provider").drop_duplicates("address_key", keep="last")
    joined = facility.merge(ok_cache, on="address_key", how="left")
    valid = joined["lon"].notna() & joined["lat"].notna()
    joined["grid_id"] = ""
    joined["x"] = np.nan
    joined["y"] = np.nan
    if valid.any():
        x, y = lonlat_to_kgd2002_central(
            pd.to_numeric(joined.loc[valid, "lon"], errors="coerce").to_numpy(),
            pd.to_numeric(joined.loc[valid, "lat"], errors="coerce").to_numpy(),
        )
        x_min = np.floor(x / grid_m) * grid_m
        y_min = np.floor(y / grid_m) * grid_m
        joined.loc[valid, "x"] = x
        joined.loc[valid, "y"] = y
        joined.loc[valid, "grid_id"] = [
            lookup.get((int(round(xm)), int(round(ym))), "") for xm, ym in zip(x_min, y_min)
        ]

    matched = joined[joined["grid_id"].astype(str).ne("")].copy()
    features = grid[["grid_id"]].copy()
    if not matched.empty:
        total = matched.groupby("grid_id").size().rename("address_facility_total").reset_index()
        features = features.merge(total, on="grid_id", how="left")
        source_pivot = (
            matched.pivot_table(index="grid_id", columns="source_id", values="source_row", aggfunc="count", fill_value=0)
            .rename(columns=lambda v: f"{safe_name(v)}_count")
            .reset_index()
        )
        features = features.merge(source_pivot, on="grid_id", how="left")

        for source_id in sorted(matched["source_id"].dropna().unique()):
            sub = matched[matched["source_id"].eq(source_id) & matched["category"].fillna("").astype(str).ne("")]
            if sub.empty:
                continue
            pivot = (
                sub.pivot_table(index="grid_id", columns="category", values="source_row", aggfunc="count", fill_value=0)
                .rename(columns=lambda v: f"{safe_name(source_id)}_{safe_name(v)}")
                .reset_index()
            )
            features = features.merge(pivot, on="grid_id", how="left")

        for src_col, out_col in [
            ("학생수", "school_student_sum"),
            ("학급수", "school_class_sum"),
            ("병상수", "shared_medical_bed_sum"),
        ]:
            if src_col in matched.columns:
                nums = pd.to_numeric(matched[src_col], errors="coerce").fillna(0)
                sums = matched.assign(_num=nums).groupby("grid_id")["_num"].sum().rename(out_col).reset_index()
                features = features.merge(sums, on="grid_id", how="left")

        for flag_col, out_col in [
            ("CT", "shared_medical_ct_available_count"),
            ("MRI", "shared_medical_mri_available_count"),
        ]:
            if flag_col in matched.columns:
                flags = matched[flag_col].astype(str).str.upper().str.strip().isin(["O", "Y", "1", "TRUE"])
                sums = matched.assign(_flag=flags.astype(int)).groupby("grid_id")["_flag"].sum().rename(out_col).reset_index()
                features = features.merge(sums, on="grid_id", how="left")

    for col in features.columns:
        if col != "grid_id":
            features[col] = pd.to_numeric(features[col], errors="coerce").fillna(0)

    summary = pd.DataFrame(
        [
            {
                "source_id": "address_sources",
                "status": "processed",
                "facility_rows": int(len(facility)),
                "unique_addresses": int(facility["address_key"].nunique()) if not facility.empty else 0,
                "geocoded_ok_addresses": int(ok_cache["address_key"].nunique()),
                "geocoded_facility_rows": int(valid.sum()),
                "matched_grid_rows": int(matched.shape[0]),
                "unmatched_grid_rows": int(valid.sum() - matched.shape[0]),
            }
        ]
    )
    return features, joined, summary


def write_outputs(
    grid_m: int,
    grid: pd.DataFrame,
    coordinate_features: pd.DataFrame,
    coordinate_assignments: pd.DataFrame,
    coordinate_summary: pd.DataFrame,
    address_features: pd.DataFrame,
    address_assignments: pd.DataFrame,
    address_summary: pd.DataFrame,
) -> dict[str, object]:
    prefix = OUTPUT_DIR / f"cheonan_grid_{grid_m}m"
    grid_csv = Path(f"{prefix}_cells_center_in.csv")
    coord_features_csv = Path(f"{prefix}_coordinate_features.csv")
    coord_assignments_csv = Path(f"{prefix}_coordinate_point_assignments.csv")
    address_features_csv = Path(f"{prefix}_address_features.csv")
    address_assignments_csv = Path(f"{prefix}_address_point_assignments.csv")
    final_csv = Path(f"{prefix}_final_features_with_addresses.csv")
    final_xlsx = Path(f"{prefix}_final_dataset_with_addresses.xlsx")
    summary_csv = Path(f"{prefix}_source_summary.csv")
    metadata_json = Path(f"{prefix}_metadata.json")

    final = grid.merge(coordinate_features, on="grid_id", how="left").merge(address_features, on="grid_id", how="left")
    for col in final.columns:
        if col not in set(grid.columns) | {"grid_id"}:
            final[col] = pd.to_numeric(final[col], errors="coerce").fillna(0)

    source_summary = pd.concat([coordinate_summary, address_summary], ignore_index=True, sort=False)

    grid.to_csv(grid_csv, index=False, encoding="utf-8-sig")
    coordinate_features.to_csv(coord_features_csv, index=False, encoding="utf-8-sig")
    coordinate_assignments.to_csv(coord_assignments_csv, index=False, encoding="utf-8-sig")
    address_features.to_csv(address_features_csv, index=False, encoding="utf-8-sig")
    address_assignments.to_csv(address_assignments_csv, index=False, encoding="utf-8-sig")
    final.to_csv(final_csv, index=False, encoding="utf-8-sig")
    source_summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(final_xlsx, engine="openpyxl") as writer:
        final.to_excel(writer, sheet_name="grid_features", index=False)
        source_summary.to_excel(writer, sheet_name="source_summary", index=False)
        coordinate_assignments.head(200000).to_excel(writer, sheet_name="coord_assignments", index=False)
        address_assignments.head(200000).to_excel(writer, sheet_name="address_assignments", index=False)

    metadata = {
        "grid_m": grid_m,
        "grid_rows": int(len(grid)),
        "final_shape": [int(final.shape[0]), int(final.shape[1])],
        "coordinate_matched_rows": int((coordinate_assignments.get("grid_id", pd.Series(dtype=str)).fillna("").astype(str).ne("")).sum())
        if not coordinate_assignments.empty
        else 0,
        "address_matched_rows": int((address_assignments.get("grid_id", pd.Series(dtype=str)).fillna("").astype(str).ne("")).sum())
        if not address_assignments.empty
        else 0,
        "outputs": {
            "grid_csv": str(grid_csv),
            "final_csv": str(final_csv),
            "final_xlsx": str(final_xlsx),
            "source_summary_csv": str(summary_csv),
            "coordinate_assignments_csv": str(coord_assignments_csv),
            "address_assignments_csv": str(address_assignments_csv),
        },
    }
    metadata_json.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    return metadata


def run_pipeline(grid_m: int, gimi9_xlsx: Path | None = None) -> dict[str, object]:
    shapes = read_boundary_shapes(BOUNDARY_ZIP)
    grid = create_grid(grid_m, shapes)
    coord_features, coord_assignments, coord_summary = aggregate_coordinate_sources(grid, grid_m)
    addr_features, addr_assignments, addr_summary = aggregate_address_sources(grid, grid_m, gimi9_xlsx)
    return write_outputs(
        grid_m,
        grid,
        coord_features,
        coord_assignments,
        coord_summary,
        addr_features,
        addr_assignments,
        addr_summary,
    )


def main() -> None:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    parser = argparse.ArgumentParser()
    parser.add_argument("--grid-m", type=int, default=750)
    parser.add_argument("--gimi9-xlsx", type=Path, default=None)
    args = parser.parse_args()
    metadata = run_pipeline(args.grid_m, args.gimi9_xlsx)
    print(json.dumps(metadata, ensure_ascii=False, indent=2))




## 3. 실행

아래 셀을 실행하면 `outputs` 폴더에 `cheonan_grid_{GRID_M}m_...` 결과물이 생성


In [ ]:
metadata = run_pipeline(GRID_M, GIMI9_XLSX)
metadata


## 4. 결과 확인

In [ ]:
import pandas as pd

final_path = OUTPUT_DIR / f"cheonan_grid_{GRID_M}m_final_features_with_addresses.csv"
summary_path = OUTPUT_DIR / f"cheonan_grid_{GRID_M}m_source_summary.csv"

final = pd.read_csv(final_path, encoding="utf-8-sig")
summary = pd.read_csv(summary_path, encoding="utf-8-sig")

print(final.shape)
display(summary)

check_cols = [
    'commerce_total', 'hospital_total', 'pharmacy_total', 'address_facility_total',
    'school_count', 'elderly_welfare_count', 'admin_agency_count', 'good_price_shop_count'
]
check_cols = [c for c in check_cols if c in final.columns]
display(final[check_cols].sum().to_frame('sum'))

show_cols = [c for c in ['grid_id', 'center_adm_nm', 'commerce_total', 'hospital_total', 'pharmacy_total', 'address_facility_total', 'school_count'] if c in final.columns]
display(final[show_cols].sort_values('commerce_total', ascending=False).head(10))
